# 📊 Home Credit - Baseline Scorecard Model (Logistic Regression + WoE / IV)

## 📌 Mục Tiêu Xây Dựng Baseline Theo Chuẩn Ngành Ngân Hàng
Theo quy trình machine learning bắt buộc trong `AGENTS.md` (Bước 5):
1. **Tính toán Weight of Evidence (WoE) & Information Value (IV)** cho tất cả các đặc trưng để đánh giá sức mạnh phân tách nợ xấu.
2. **Lọc Feature theo chuẩn IV & Multicollinearity**: Giữ lại các biến có $IV \ge 0.02$ và loại bỏ các biến đa cộng tuyến ($|r| > 0.8$).
3. **Chống Data Leakage**: Tách tập Train (80%) và Validation (20%) bằng Stratified K-Fold trước mọi thao tác Scaler & Imputation.
4. **Huấn luyện Baseline Scorecard**: Sử dụng thuật toán **Logistic Regression** với `class_weight='balanced'`.
5. **Thước đo Đánh giá Ngân hàng**: Tính toán **ROC-AUC**, **PR-AUC**, **KS Statistic** (Kolmogorov-Smirnov), **Gini Coefficient**, và **Calibration Curve** (Brier Score).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11

DATA_PATH = Path('../data/processed/home_credit_processed.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/home_credit_processed.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f'✓ Nạp dữ liệu thành công từ {DATA_PATH}: shape = {df.shape}')
else:
    RAW_PATH = Path('../data/raw/home-credit-default-risk/application_train.csv')
    if not RAW_PATH.exists():
        RAW_PATH = Path('data/raw/home-credit-default-risk/application_train.csv')
    df = pd.read_csv(RAW_PATH, nrows=50000)
    df['CREDIT_TO_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_TO_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df = pd.get_dummies(df, drop_first=True)

---
## 1. 🔍 Tính Toán Weight of Evidence (WoE) & Information Value (IV) - Lọc Feature

In [ ]:
def calculate_iv(df, target_col, bins=10):
    iv_list = []
    features = [c for c in df.columns if c not in [target_col, 'SK_ID_CURR']]
    
    total_goods = (df[target_col] == 0).sum()
    total_bads = (df[target_col] == 1).sum()
    
    for feat in features:
        try:
            if df[feat].nunique() <= 2:
                binned = df[feat]
            else:
                binned = pd.qcut(df[feat], q=bins, duplicates='drop')
            
            grouped = df.groupby(binned, observed=False)[target_col].agg(['count', 'sum'])
            grouped['goods'] = grouped['count'] - grouped['sum']
            grouped['bads'] = grouped['sum']
            
            grouped['dist_goods'] = (grouped['goods'] + 0.5) / (total_goods + 1)
            grouped['dist_bads'] = (grouped['bads'] + 0.5) / (total_bads + 1)
            
            grouped['woe'] = np.log(grouped['dist_goods'] / grouped['dist_bads'])
            grouped['iv'] = (grouped['dist_goods'] - grouped['dist_bads']) * grouped['woe']
            
            total_iv = grouped['iv'].sum()
            iv_list.append({'Feature': feat, 'IV': total_iv})
        except Exception:
            continue
            
    iv_df = pd.DataFrame(iv_list).sort_values(by='IV', ascending=False)
    return iv_df

print('► Bắt đầu tính toán Information Value (IV) cho tập dữ liệu...')
sample_for_iv = df.sample(n=min(30000, len(df)), random_state=42)
iv_summary = calculate_iv(sample_for_iv, 'TARGET')
print('=== TOP 15 ĐẶC TRƯNG CÓ INFORMATION VALUE (IV) CAO NHẤT ===')
print(iv_summary.head(15).to_string(index=False))

# Sàng lọc các biến có IV >= 0.02 (Loại bỏ Uninformative Features)
selected_iv_features = iv_summary[iv_summary['IV'] >= 0.02]['Feature'].tolist()
print(f'✓ Số lượng đặc trưng giữ lại sau lọc IV (IV >= 0.02): {len(selected_iv_features)} / {len(df.columns)-2}')

---
## 2. ✂️ Tách Dữ Liệu Train / Validation & Pipeline Imputation (Anti Data Leakage)

In [ ]:
# Tách X và y chỉ lấy các biến đã qua sàng lọc IV
X = df[selected_iv_features] if len(selected_iv_features) > 5 else df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
y = df['TARGET']

# Chia Stratified Train / Validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'► Tập Train: {X_train.shape}, Tỷ lệ Target = {y_train.mean():.4f}')
print(f'► Tập Val:   {X_val.shape}, Tỷ lệ Target = {y_val.mean():.4f}')

# Imputation & Scaling fit CHỈ TRÊN TẬP TRAIN để tránh Data Leakage
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)

print('✓ Đã hoàn tất Imputation & Feature Scaling chuẩn hóa.')

---
## 3. 🤖 Huấn Luyện Baseline Logistic Regression Scorecard

In [ ]:
# Huấn luyện Logistic Regression với class_weight='balanced' do mất cân bằng dữ liệu
lr_model = LogisticRegression(class_weight='balanced', C=0.05, max_iter=500, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_proba_train = lr_model.predict_proba(X_train_scaled)[:, 1]
y_pred_proba_val = lr_model.predict_proba(X_val_scaled)[:, 1]

train_auc = roc_auc_score(y_train, y_pred_proba_train)
val_auc = roc_auc_score(y_val, y_pred_proba_val)
print(f'🎯 Train ROC-AUC: {train_auc:.4f}')
print(f'🎯 Validation ROC-AUC: {val_auc:.4f}')

---
## 4. 📈 Đánh Giá Chỉ Số Rủi Ro Tín Dụng (ROC-AUC, PR-AUC, KS, Gini, Calibration)

In [ ]:
def calculate_credit_metrics(y_true, y_prob):
    auc_score = roc_auc_score(y_true, y_prob)
    gini_score = 2 * auc_score - 1
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ks_stat = np.max(tpr - fpr)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    brier = brier_score_loss(y_true, y_prob)
    return {
        'ROC-AUC': auc_score,
        'Gini': gini_score,
        'KS Statistic (%)': ks_stat * 100,
        'PR-AUC': pr_auc,
        'Brier Score': brier
    }

metrics_lr = calculate_credit_metrics(y_val, y_pred_proba_val)
print('=== KẾT QUẢ ĐÁNH GIÁ BASELINE LOGISTIC REGRESSION ===')
for k, v in metrics_lr.items():
    print(f'► {k:20s}: {v:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_val, y_pred_proba_val)
axes[0].plot(fpr, tpr, label=f'Logistic Regression (AUC = {metrics_lr["ROC-AUC"]:.3f})', color='#2a9d8f', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[0].set_title('Đường Cong ROC (ROC Curve)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR)')
axes[0].legend(loc='lower right')

ks_idx = np.argmax(tpr - fpr)
axes[1].plot(fpr, label='FPR (Tỷ lệ báo động nhầm)', color='#2a9d8f')
axes[1].plot(tpr, label='TPR (Tỷ lệ bắt nợ xấu)', color='#e76f51')
axes[1].set_title(f'Biểu Đồ Kolmogorov-Smirnov (KS = {metrics_lr["KS Statistic (%)"]:.1f}%)', fontweight='bold')
axes[1].set_xlabel('Threshold Index')
axes[1].set_ylabel('Rate')
axes[1].legend()

plt.tight_layout()
plt.show()